In [51]:
import pandas as pd


df_inds = pd.read_csv("test_2mobj/statistics/run15_individuals.csv")
df_inds[['dG_separated/dSASAx100', 'f_attr']] = df_inds['fitness'].str.split(',', expand=True)
df_inds.drop(columns=['fitness', 'pdb_file', 'father', 'id', 'generation'], inplace=True)
df_inds

,nmut,sequence,dG_separated/dSASAx100,f_attr
0,0,GYSYNTSVSGGSYNMDYLNRTSYECFNGSNFYSGVGPT,-2.05,-1305.021
1,1,GYSYNTSVSGGSYNMDYLNRTSYECFHGSNFYSGVGPT,-2.029,-1304.191
2,1,GYSYNTSVSGGSYNMDYLNRTSYECFHGSNFYSGVGPT,-2.029,-1304.191
3,1,GYSYNTSVSGGSYNMEYLNRTSYECFNGSNFYSGVGPT,-2.29,-1307.606
4,1,GYSYNTSVSGGSYNMEYLNRTSYECFNGSNFYSGVGPT,-2.289,-1307.606
...,...,...,...,...
295,8,RYSYNNSVSGGSYTMEYLNQTSGECFTGSNFYSGVGLT,-2.644,-1316.042
296,9,RYSYNQSVSGGSYKMEYLNGTNFECINGSNFYSGVGLT,-2.371,-1323.631
297,6,RYSYNKSVSGGSYHMEYLNLTSYECFNGSNFYSGVGLT,-2.269,-1319.286
298,9,RYSYNASVSGGSYKMEYLNGTNVECINGSNFYSGVGFT,-2.302,-1314.024


In [52]:
import numpy as np
import pandas as pd

# 1) Alfabeto (ajústalo si tienes símbolos extra)
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")  # 20 estándar
UNK = "X"                                   # token para desconocidos
ALPH = AMINO_ACIDS + [UNK]
A = len(ALPH)

# 2) Secuencias
seqs = df_inds["sequence"].astype(str).tolist()
L = len(seqs[0])
assert all(len(s) == L for s in seqs), "Todas las secuencias deben tener la misma longitud"

# 3) Mapear letras -> índices
idx = {aa:i for i,aa in enumerate(ALPH)}

# 4) Convertir a matriz (N, L) de índices
#    Si aparece una letra fuera del alfabeto, la mandamos a UNK
def seq_to_idx(s):
    return [idx.get(ch, idx[UNK]) for ch in s]

arr_idx = np.array([seq_to_idx(s) for s in seqs], dtype=int)  # (N, L)
N = arr_idx.shape[0]

# 5) One-hot: (N, L, A) -> (N, L*A)
one_hot = np.eye(A, dtype=int)[arr_idx]        # (N, L, A)
one_hot_flat = one_hot.reshape(N, L*A)         # (N, L*A)

# 6) Columnas
cols = [f"pos{pos+1}_{aa}" for pos in range(L) for aa in ALPH]

# 7) DataFrame
df_onehot = pd.DataFrame(one_hot_flat, columns=cols, index=df_inds.index)

# 8) Unir con tus métricas
df_encoded = pd.concat([df_inds[['dG_separated/dSASAx100', 'f_attr']].reset_index(drop=True), df_onehot.reset_index(drop=True)], axis=1)
df_encoded

,dG_separated/dSASAx100,f_attr,pos1_A,pos1_C,pos1_D,pos1_E,pos1_F,pos1_G,pos1_H,pos1_I,...,pos38_N,pos38_P,pos38_Q,pos38_R,pos38_S,pos38_T,pos38_V,pos38_W,pos38_Y,pos38_X
0,-2.05,-1305.021,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
1,-2.029,-1304.191,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
2,-2.029,-1304.191,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,-2.29,-1307.606,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,-2.289,-1307.606,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,-2.644,-1316.042,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
296,-2.371,-1323.631,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
297,-2.269,-1319.286,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
298,-2.302,-1314.024,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [65]:
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder

# Preparamos los datos
X = df_encoded.drop(columns=['dG_separated/dSASAx100', 'f_attr'])
y_fattr = df_encoded['f_attr'].astype(float)
y_dg = df_encoded['dG_separated/dSASAx100'].astype(float)

# Modelo para f_attr
xgb_fattr = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_fattr.fit(X, y_fattr)
importances_fattr = pd.Series(xgb_fattr.feature_importances_, index=X.columns).sort_values(ascending=False)

# Modelo para dG_separated/dSASAx100
xgb_dg = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_dg.fit(X, y_dg)
importances_dg = pd.Series(xgb_dg.feature_importances_, index=X.columns).sort_values(ascending=False)

importances_fattr.head(15), importances_dg.head(15)

(pos1_R     0.704617
 pos23_Y    0.043299
 pos22_S    0.034194
 pos6_R     0.023475
 pos2_C     0.021598
 pos27_Y    0.019995
 pos20_P    0.019593
 pos23_V    0.017916
 pos27_T    0.011390
 pos20_H    0.010473
 pos6_M     0.007996
 pos37_L    0.007592
 pos6_G     0.006663
 pos23_C    0.005103
 pos23_S    0.004862
 dtype: float32,
 pos6_N     0.362487
 pos16_D    0.165652
 pos22_N    0.055253
 pos20_Q    0.046484
 pos27_Y    0.045243
 pos30_L    0.041545
 pos30_H    0.019875
 pos6_G     0.015372
 pos16_S    0.013361
 pos14_N    0.012980
 pos14_H    0.011825
 pos27_D    0.011788
 pos23_S    0.011599
 pos38_S    0.011455
 pos16_E    0.010659
 dtype: float32)